# LeNet with arithmetic symbols
extension option 2: recognise + - * / as well as digits, then calculate the result

starts from the mixed model, swaps the 10 digit output layer for 14 outputs (0-9 and + - * /) and fine tunes it.
symbols come from the Handwritten Math Symbols dataset (Kaggle, sagyamthapa). brackets aren't in the dataset yet

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from hnrs.swiss_digits import load_swiss_digits
from hnrs.preprocessing import load_preprocessed_mnist
from hnrs.math_symbols import load_math_symbols, CLASSES
from hnrs.models.lenet import fine_tune_symbol_model
from hnrs.arithmetic import calculate

math_train, math_train_y, math_test, math_test_y = load_math_symbols()
swiss_train, swiss_train_y, swiss_test, swiss_test_y = load_swiss_digits()
mnist_train, mnist_train_y, _, mnist_test, mnist_test_y, _ = load_preprocessed_mnist()

print("math:", math_train.shape, math_test.shape)
print("classes:", CLASSES)

## 1. build the training mix
symbols only have about 440 training images each. if we only train on the symbols dataset the model forgets swiss and mnist handwriting.
so we add the same number of swiss and mnist images per digit

In [ ]:
rng = np.random.default_rng(42)
per_class = int(np.bincount(math_train_y).mean())

def sample_digits(x, y, n):
    """
    returns n random images of each digit 0-9
    """
    idx = np.concatenate([rng.choice(np.where(y == d)[0], n, replace=False) for d in range(10)])
    return x[idx], y[idx]

sx, sy = sample_digits(swiss_train, swiss_train_y, per_class)
mx, my = sample_digits(mnist_train, mnist_train_y, per_class)

x = np.concatenate([math_train, sx, mx])
y = np.concatenate([math_train_y, sy, my])
order = rng.permutation(len(x))
x, y = x[order], y[order]

print(f"{per_class} per class, {len(x)} training images")

## 2. fine tune

In [ ]:
model, history = fine_tune_symbol_model(x, y, num_classes=len(CLASSES), epochs=5)

## 3. evaluate on symbols dataset test set

In [ ]:
pred = model.predict(math_test, verbose=0).argmax(axis=1)
print(classification_report(math_test_y, pred, target_names=CLASSES, digits=3))

ConfusionMatrixDisplay.from_predictions(math_test_y, pred, display_labels=CLASSES, colorbar=False)
plt.title("symbol model confusion matrix")
plt.show()

## 4. check digits werent forgotten
compare with the mixed model results from lenet_transfer.ipynb.
also counts how often a digit gets mistaken for a symbol

In [ ]:
for name, xt, yt in [("swiss", swiss_test, swiss_test_y), ("mnist", mnist_test, mnist_test_y)]:
    pred = model.predict(xt, verbose=0).argmax(axis=1)
    print(f"{name}: {np.mean(pred == yt) * 100:.1f}% correct, {np.mean(pred >= 10) * 100:.1f}% predicted as a symbol")

## 5. calculate expressions
picks a random test image for each character, predicts each one and calculates the result.
this skips segmentation, so it only tests recognition + calculation

In [ ]:
def pick_image(char):
    """
    returns a random test image of the given character
    """
    label = CLASSES.index(char)
    return math_test[rng.choice(np.where(math_test_y == label)[0])]

for expression in ["12+3*4", "9-4/2", "7*8", "100/25+6"]:
    images = np.array([pick_image(c) for c in expression])
    predicted = "".join(CLASSES[i] for i in model.predict(images, verbose=0).argmax(axis=1))
    try:
        result = calculate(predicted)
    except ValueError as e:
        result = f"error: {e}"
    print(f"actual {expression} = {calculate(expression)} | predicted {predicted} = {result}")

    plt.figure(figsize=(len(expression), 1.2))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(img, cmap="gray")
        plt.axis("off")
    plt.show()